In [ ]:
# ============================================================
# CELL 1: SETUP & INSTALLATION
# ============================================================

from google.colab import drive

# Mount Google Drive so that the Hindi and Bengali recordings
# can be accessed directly from the dataset directories.
drive.mount('/content/drive')

# Install the same core dependencies used in the original
# four-language synthetic voice conversion notebook.
!pip install -q librosa soundfile scipy tqdm
!apt -qq install -y ffmpeg

print("✅ Dependencies installed!")

In [ ]:
# ============================================================
# CELL 2: LANGUAGE CONFIGURATIONS
# ============================================================

import os

LANGUAGES = {

    "hindi": {
        "input": "/content/drive/MyDrive/five_songs_dataset/hindi",
        "output": "/content/drive/MyDrive/five_language_output/hindi/sovits"
    },

    "bengali": {
        "input": "/content/drive/MyDrive/five_songs_dataset/bengali",
        "output": "/content/drive/MyDrive/five_language_output/bengali/sovits"
    }
}

# These are the same three synthetic voice configurations
# used in the original notebook.
CHARACTERS = [
    "kawaii_girl",
    "energetic_idol",
    "soft_anime"
]

print("✅ Configured languages:")

for lang in LANGUAGES.keys():
    print(f"  • {lang.title()}")

print("\n🎭 Voice configurations:")

for character in CHARACTERS:
    print(f"  • {character}")

In [ ]:
# ============================================================
# CELL 3: CREATE OUTPUT DIRECTORIES
# ============================================================

import os

for lang, paths in LANGUAGES.items():

    # Store intermediate WAV outputs here.
    os.makedirs(
        f"{paths['output']}/output",
        exist_ok=True
    )

    # Store final MP3 files here.
    os.makedirs(
        f"{paths['output']}/FINAL_COLLECTION",
        exist_ok=True
    )

    print(
        f"✅ Created folders for {lang.title()}"
    )

print("\n✅ All output directories ready!")

In [ ]:
# ============================================================
# CELL 4: SYNTHETIC VOICE CONVERTER
# ============================================================

import librosa
import numpy as np
import soundfile as sf


class AnimeVoiceConverter:

    def __init__(self, char_name):

        self.char_name = char_name

        # Use the exact voice transformation settings
        # defined in the original notebook.
        self.settings = {

            "kawaii_girl": {
                "pitch": +4,
                "formant": 0.8,
                "bright": 1.3,
                "breath": 0.7
            },

            "energetic_idol": {
                "pitch": +5,
                "formant": 1.0,
                "bright": 1.4,
                "breath": 0.5
            },

            "soft_anime": {
                "pitch": +3,
                "formant": 0.6,
                "bright": 1.1,
                "breath": 0.9
            }

        }[char_name]


    def _anime_fx(self, y, sr):

        # ----------------------------------------------------
        # STEP 1: PITCH SHIFT
        # ----------------------------------------------------
        # Shift the singing voice upward according to the
        # selected synthetic voice configuration.
        y = librosa.effects.pitch_shift(
            y,
            sr=sr,
            n_steps=self.settings["pitch"]
        )

        # ----------------------------------------------------
        # STEP 2: STFT
        # ----------------------------------------------------
        # Convert the audio into its time-frequency
        # representation for spectral manipulation.
        S = librosa.stft(
            y,
            n_fft=2048,
            hop_length=512
        )

        magnitude = np.abs(S)
        phase = np.angle(S)

        # Generate the frequency bins corresponding to
        # the STFT representation.
        frequencies = librosa.fft_frequencies(
            sr=sr,
            n_fft=2048
        )

        # ----------------------------------------------------
        # STEP 3: SPECTRAL BRIGHTENING
        # ----------------------------------------------------
        # Increase the 1–4 kHz region to produce a brighter
        # synthetic voice characteristic.
        magnitude[
            (frequencies > 1e3) &
            (frequencies < 4e3)
        ] *= self.settings["bright"]

        # ----------------------------------------------------
        # STEP 4: HIGH-FREQUENCY MODIFICATION
        # ----------------------------------------------------
        # Modify frequencies above 6 kHz according to the
        # selected breathiness setting.
        magnitude[
            frequencies > 6e3
        ] *= self.settings["breath"]

        # ----------------------------------------------------
        # STEP 5: STFT RECONSTRUCTION
        # ----------------------------------------------------
        # Recombine the modified magnitude with the original
        # phase information.
        y = librosa.istft(
            magnitude * np.exp(1j * phase),
            hop_length=512
        )

        # Normalize and apply the same soft non-linear
        # amplitude transformation used in the source code.
        return librosa.util.normalize(
            np.tanh(y * 1.2) * 0.9
        )


    def convert(self, in_path, out_path):

        try:

            # The original pipeline processes audio at 44.1 kHz
            # and converts it to mono before transformation.
            y, sr = librosa.load(
                in_path,
                sr=44100,
                mono=True
            )

            # Apply the synthetic voice transformation.
            y_synthetic = self._anime_fx(
                y,
                sr
            )

            # Save the generated synthetic singing voice.
            sf.write(
                out_path,
                y_synthetic,
                sr
            )

            return True

        except Exception as e:

            print(
                f"    ⚠️ Error: {e}"
            )

            return False


# Create one converter instance for each synthetic
# voice configuration.
converters = {
    character: AnimeVoiceConverter(character)
    for character in CHARACTERS
}

print(
    "✅ Synthetic voice converters initialized!"
)

In [ ]:
# ============================================================
# CELL 5: BATCH CONVERT HINDI AND BENGALI
# ============================================================

import os
from pathlib import Path
from tqdm import tqdm


def pick_char(idx):
    """
    Rotate between the three synthetic voice configurations.

    Example:
        0 → kawaii_girl
        1 → energetic_idol
        2 → soft_anime
        3 → kawaii_girl
        ...
    """

    return CHARACTERS[
        idx % len(CHARACTERS)
    ]


total_converted = 0
results_summary = {}


for lang, paths in LANGUAGES.items():

    print("\n" + "=" * 60)
    print(
        f"🎵 PROCESSING {lang.upper()} SONGS"
    )
    print("=" * 60)

    input_dir = paths["input"]
    output_dir = paths["output"]

    # Check whether the language input directory exists.
    if not os.path.exists(input_dir):

        print(
            f"⚠️ Skipping {lang} - "
            f"input folder not found: {input_dir}"
        )

        results_summary[lang] = {
            "total": 0,
            "success": 0,
            "failed": 0
        }

        continue

    # Collect supported audio recordings.
    songs = sorted([
        f for f in os.listdir(input_dir)
        if f.lower().endswith(
            (
                ".wav",
                ".mp3",
                ".flac",
                ".ogg",
                ".m4a"
            )
        )
    ])

    if len(songs) == 0:

        print(
            f"⚠️ No audio files found in {input_dir}"
        )

        results_summary[lang] = {
            "total": 0,
            "success": 0,
            "failed": 0
        }

        continue

    print(
        f"📂 Found {len(songs)} songs"
    )

    successful = 0
    failed = 0

    # Process every recording in the selected language.
    for idx, file in enumerate(
        tqdm(
            songs,
            desc=f"{lang.title()}"
        )
    ):

        # Rotate through the three synthetic voice styles.
        character = pick_char(idx)

        input_path = os.path.join(
            input_dir,
            file
        )

        clean_name = Path(file).stem

        # Keep the language, synthetic voice configuration,
        # original index and source filename in the output name.
        output_path = (
            f"{output_dir}/output/"
            f"anime_{character}_"
            f"{idx + 1:03d}_"
            f"{clean_name}.wav"
        )

        if converters[character].convert(
            input_path,
            output_path
        ):

            successful += 1

        else:

            failed += 1

    total_converted += successful

    results_summary[lang] = {
        "total": len(songs),
        "success": successful,
        "failed": failed
    }

    print(
        f"✅ {lang.title()}: "
        f"{successful}/{len(songs)} "
        f"converted successfully"
    )

    if failed > 0:

        print(
            f"⚠️ {failed} files failed"
        )


print("\n" + "=" * 60)
print(
    f"🎉 TOTAL: {total_converted} "
    f"songs converted across "
    f"{len(LANGUAGES)} languages!"
)
print("=" * 60)

In [ ]:
# ============================================================
# CELL 6: CONVERT ALL WAV FILES TO MP3
# ============================================================

from pathlib import Path
import subprocess


print(
    "🎵 Converting all WAV files to MP3 (320 kbps)...\n"
)


for lang, paths in LANGUAGES.items():

    output_dir = Path(
        f"{paths['output']}/output"
    )

    final_dir = Path(
        f"{paths['output']}/FINAL_COLLECTION"
    )

    if not output_dir.exists():
        continue

    wav_files = list(
        output_dir.glob("*.wav")
    )

    if len(wav_files) == 0:

        print(
            f"⚠️ {lang.title()}: "
            f"No WAV files to convert"
        )

        continue

    print(
        f"📀 {lang.title()}: "
        f"Converting {len(wav_files)} files..."
    )

    converted = 0

    for wav_file in wav_files:

        mp3_file = (
            final_dir /
            wav_file.name.replace(
                ".wav",
                ".mp3"
            )
        )

        # FFmpeg is used to create the final 320 kbps MP3
        # collection, matching the original notebook.
        result = subprocess.run(
            [
                "ffmpeg",
                "-loglevel",
                "quiet",
                "-y",
                "-i",
                str(wav_file),
                "-codec:a",
                "libmp3lame",
                "-b:a",
                "320k",
                str(mp3_file)
            ],
            capture_output=True
        )

        if result.returncode == 0:

            converted += 1

    print(
        f"  ✅ {converted}/{len(wav_files)} "
        f"MP3 files created\n"
    )


print(
    "✅ All conversions completed!"
)

In [ ]:
# ============================================================
# CELL 7: FINAL SUMMARY & STATISTICS
# ============================================================

from pathlib import Path


print("=" * 70)
print(
    "🌟 HINDI + BENGALI SYNTHETIC "
    "VOICE CONVERSION — FINAL REPORT"
)
print("=" * 70)


grand_total = 0


for lang, paths in LANGUAGES.items():

    final_dir = Path(
        f"{paths['output']}/FINAL_COLLECTION"
    )

    if not final_dir.exists():

        print(
            f"\n❌ {lang.title()}: "
            f"No output folder"
        )

        continue

    mp3_files = sorted(
        final_dir.glob("*.mp3")
    )

    total_size_mb = (
        sum(
            f.stat().st_size
            for f in mp3_files
        )
        / (1024 * 1024)
    )

    grand_total += len(mp3_files)

    print(
        f"\n📊 {lang.upper()}"
    )

    print(
        f"  📁 Location: "
        f"{paths['output']}/FINAL_COLLECTION"
    )

    print(
        f"  🎵 Songs: "
        f"{len(mp3_files)}"
    )

    print(
        f"  💾 Size: "
        f"{total_size_mb:.1f} MB"
    )

    # Count outputs generated using each synthetic
    # voice configuration.
    print(
        "  🎭 Voice configurations:"
    )

    for character in CHARACTERS:

        count = len([
            f for f in mp3_files
            if character in f.name
        ])

        print(
            f"     • {character}: "
            f"{count} songs"
        )


print("\n" + "=" * 70)
print(
    f"🎉 GRAND TOTAL: "
    f"{grand_total} synthetic recordings"
)
print("=" * 70)